In [1]:
import os 
# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/tabula_muris_brain/Leaflet


In [2]:
import sys
sys.path.append('../../utils')
from functions import * 

In [ ]:
from importlib import reload
import sys

# Path to the Leaflet repository
PATH_TO_LEAFLET_REPO = '/gpfs/commons/home/kisaev/Leaflet/src/beta-binomial-mix/'
sys.path.append(PATH_TO_LEAFLET_REPO)

In [ ]:
from importlib import reload
from load_cluster_data import load_cluster_data
from betabinomo_mix_singlecells import *
import betabinomo_mix_singlecells
reload(betabinomo_mix_singlecells)
from cell_state_asign_consistency import *
#reload(cell_state_asign_consistency)
import torch
import sklearn.manifold 
import plotnine as p9
import time
# indicate plot should be small 4 by 4
import plotnine as p9
from plotnine import ggplot, geom_point, aes, stat_smooth, facet_wrap, geom_violin, theme, element_blank, geom_text
import plotnine
from tqdm import tqdm
plotnine.options.figure_size = (4, 4)
import seaborn as sns
sns.set_theme(style="whitegrid")

### Settings and Load data

In [ ]:
torch.manual_seed(42)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

float_type = { 
    "device" : device, 
    "dtype" : torch.float, # save memory
}

hypers = {
    "eta" : 1., 
    "alpha_prior" : 1., # karin had 0.65 
    "pi_prior" : 1.
}

### load_cluster_data takes ~ 5 minutes for ss2 brain data.... 

In [ ]:
# this folder contains input data for each tissue cell type sample
input_files_folder = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/Brain/train/'

final_data, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = load_cluster_data(
    input_folder = input_files_folder) 

In [ ]:
# validation data 
input_files_folder = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/Brain/validation/'

val_data, val_coo_counts_sparse, val_coo_cluster_sparse, val_cell_ids_conversion, val_junction_ids_conversion = load_cluster_data(
    input_folder = input_files_folder) 

In [ ]:
# ensure that in coo_counts_sparse.shape = (n_cells, n_genes) , n_cells is the same number as cell_ids_conversion.shape = (n_cells, num_variables)
assert coo_counts_sparse.shape[0] == cell_ids_conversion.shape[0]

In [ ]:
cell_index_tensor, junc_index_tensor, my_data = betabinomo_mix_singlecells.make_torch_data(final_data, **float_type)

In [ ]:
# set random seed
torch.manual_seed(0)

num_trials = 10 # should also be an argument that gets fed in
num_iters = 50 # should also be an argument that gets fed in
K = 20

# loop over the number of trials (for now just testing using one trial but in general need to evaluate how performance is affected by number of trials)
reload(betabinomo_mix_singlecells)

start_time = time.time()
all_results_k = []

for k in range(K):
    k = k + 1
    print(f"Running with {k} cell states")
    results = [ betabinomo_mix_singlecells.calculate_CAVI(k, my_data, float_type, hypers, init_labels = None, num_iterations = num_iters) 
           for t in range(num_trials) ]
    all_results_k.append(results)

# write the above line use fstring
print(f"This took {time.time() - start_time} seconds")

### Evaluate how consistent the cell type assignments are across the trials of the mixture model

In [ ]:
# Generate or load your list of matrices
#nonzero_dfs, zero_dfs, matrices_list =  cell_state_asign_consistency.check_cell_pairs(results)
#print(nonzero_dfs.head())

### Evaluate the learned posterions

In [ ]:
elbos_ks = []
for i in range(len(all_results_k)):
    print("Reporting ELBO for k = " + str(i+1))
    results = all_results_k[i]
    best = np.argmax([ g[-1][-1] for g in results ]) # final ELBO
    ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[best]
    elbos_all = np.array(elbos_all)
    print("ELBO: " + str(elbos_all[-1]))
    elbos_ks.append(elbos_all[-1])
    #plt.plot(elbos_all[1:]); plt.show()

In [ ]:
elbos_ks = pd.DataFrame(elbos_ks)
elbos_ks["k"] = range(K)
elbos_ks

In [ ]:
best_k = elbos_ks.sort_values(by=0, ascending=False).head(1)
print("The k with the highest ELBO is: " + str(best_k.index[0]))

In [ ]:
results = all_results_k[best_k.index[0]]

In [ ]:
best = np.argmax([ g[-1][-1] for g in results ]) # final ELBO
print(f"The trial with the highest ELBO was {best}")
ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[best]
elbos_all = np.array(elbos_all)
plt.plot(elbos_all[1:]); plt.show()

In [ ]:
K = best_k.index[0]+1 
print(K)

In [ ]:
juncs_probs = ALPHA_f / (ALPHA_f+PI_f)   
 
plt.hist(juncs_probs.cpu().numpy().flatten(), 20)
plt.title('Histogram of learned junction probabilities') 
plt.xlabel('Probability of junction success')
plt.show()

In [ ]:
PHI_f_plot = pd.DataFrame(PHI_f.cpu().numpy())
PHI_f_plot['cell_id'] = cell_ids_conversion["cell_type"].to_numpy()
PHI_f_summ = PHI_f_plot.groupby('cell_id').mean()
print(PHI_f_summ)

In [ ]:
# How much each cell state is used 
#latent proportions describe the general prevalence of each cluster in the datasetN
# cluster proportions via theta ~ dirichlet(GAMMA_f)
# each cell gets an assignment to a cluster via z_c | theta ~ categorical(theta)

theta = GAMMA_f / GAMMA_f.sum()
theta = theta.cpu().numpy()
theta_sorted = np.sort(theta)
theta_sorted

In [ ]:
PHI_f.shape #<- this is the matrix of probabilities of each cell belonging to each cluster

In [ ]:
theta_sorted

In [ ]:
x = PHI_f.cpu().numpy()
_ = plt.hist(x.flatten(),100)

In [ ]:
juncs_probs_df = pd.DataFrame(juncs_probs, columns = range(K))
# add "cell_state" to each column name 
juncs_probs_df.columns = ["cell_state_" + str(col) for col in juncs_probs_df.columns]
juncs_probs_df["junction_id_index"] = junction_ids_conversion.junction_id_index.values
# convert to juncs_probs to pandas dataframe and calculate mean and std across cell states/topics
juncs_probs_df["junction_id"] = junction_ids_conversion.junction_id.values

In [ ]:
def plot_juncObsUsage(junc_index):

    # print junction ID using junction_ids_conversion
    print(junction_ids_conversion[junction_ids_conversion["junction_id_index"] == junc_index])
    junc_id = junction_ids_conversion[junction_ids_conversion["junction_id_index"] == junc_index].junction_id.values[0]

    # get data for just junc_index 
    junc_dat=final_data[final_data.junction_id_index==junc_index]
    print(junc_dat.cell_type.value_counts())

    # make violin plot for junc_dat junction usage ratio coloured by cell_type and rotate plot 90 degrees
    plot = ggplot(junc_dat, aes(x='cell_type', y='juncratio', fill="cell_type")) + geom_violin() + geom_point() + plotnine.labels.ggtitle(junc_id) + plotnine.coords.coord_flip() 

    # add number of cells in each cell_type to plot 
    print(plot)

def plot_juncProbs(junc_index):
    
    # print junction ID using junction_ids_conversion
    print(junction_ids_conversion[junction_ids_conversion["junction_id_index"] == junc_index])
    junc_id = junction_ids_conversion[junction_ids_conversion["junction_id_index"] == junc_index].junction_id.values[0]
    
    # get data for just junc_index 
    junc_dat=juncs_probs_df[juncs_probs_df.junction_id_index==junc_index]
    junc_dat = junc_dat.melt().iloc[0:K]
    junc_dat.value = junc_dat.value.astype(float)
    # make violin plot for junc_dat junction usage ratio coloured by cell_type
    # don't print x-axis tick labels 
    plot = ggplot(junc_dat, aes(x='variable', y='value')) + geom_point() + theme(axis_text_x=element_blank())
    print(plot)

In [ ]:
# calculate sd deviation for each junction for cell states 0 to 19 
juncs_probs_df["sd"] = juncs_probs_df.iloc[:,0:K].std(axis=1)

In [ ]:
# convert PHI_f to a dataframe and add a column with cell ID and cell type 
PHI_f = pd.DataFrame(PHI_f)

# Add "CellState" to each column 
PHI_f.columns = ["CellState_" + str(i) for i in range(PHI_f.shape[1])]
PHI_f['cell_id'] = cell_ids_conversion.cell_id.values
PHI_f['cell_type'] = cell_ids_conversion.cell_type.values

In [ ]:
PHI_f.groupby('cell_type').sum()

In [ ]:
# get likelihood ratio/bayes factor score for ALL junctions 
# let's compare just state X and Y

scores_all_juncs = []
for junc_index in range(juncs_probs.shape[0]):
    a = ALPHA_f[junc_index, [3,6]]
    b = PI_f[junc_index, [3,6]]
    scores_all_juncs.append(score(a, b).item())

# turn scores_all_juncs into dataframe and add junction_id_index as a column
scores_all_juncs_df = pd.DataFrame(scores_all_juncs, columns = ["score"])
scores_all_juncs_df["junction_id_index"] = junction_ids_conversion.junction_id_index.values
scores_all_juncs_df.sort_values(by="score", ascending=False).head(10)
juncs_test=scores_all_juncs_df.sort_values(by="score", ascending=False).head(2).junction_id_index.values

In [ ]:
# for each junction in top10juncs_state1, run plot_juncObsUsage and plot_juncProbs
for junc in juncs_test:
    plot_juncObsUsage(junc)
    plot_juncProbs(junc)

In [ ]:
# save cell assignments to file for downstream analysis
output_dir = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/'

# BBmixture model cell assignments
output_file = os.path.join(output_dir, 'Leaflet_BBmixture.csv')
PHI_f.to_csv(output_file, index=True, header=True)
print('Saved Leaflet latent cell states to {}'.format(output_file))

In [ ]:
PHI_f["CellState_0"].describe()

In [ ]:
# Create a 2 columns x 5 rows grid for the subplots
fig, axes = plt.subplots(5, 4, figsize=(12, 25))
# Flatten the axes array to loop through each subplot
axes = axes.flatten()

# plot distribution of values in each CellState_X column for each cell type
for i in range(K):
    dat = PHI_f[['CellState_{}'.format(i), "cell_type"]]
    dat = dat.copy()
    dat["CellState"] = dat["CellState_{}".format(i)]
    sns.violinplot(x='cell_type', y='CellState', data=dat, ax=axes[i], scale='count')
    # reduce size of x-axis labels font
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=90, fontsize=6)
    axes[i].set_title('CellState_{}'.format(i))

# Remove any empty subplots (if K is less than 10)
for j in range(K, 20):
    fig.delaxes(axes[j])

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

In [ ]:
# group by cell_type and sum across each cellstate 
PHI_f.groupby('cell_type').sum()

In [ ]:
sum_prop=PHI_f.groupby('cell_type').sum()/PHI_f.groupby('cell_type').count()
# remove cell_id column 
sum_prop=sum_prop.drop(columns=['cell_id'])

In [ ]:
sns.heatmap(sum_prop)

In [ ]:
sum_prop

In [ ]:
sum_prop.index

In [ ]:
# now let's give the cell states new labels based on what cell types they are associated with 
# go through sum_prop and for each cell state find the cell type with the highest proportion
# assign that cell type to the cell state
cell_type_labels = {}
for state in sum_prop.columns:
    max_prop = 0
    max_celltype = ''
    for celltype in sum_prop.index:
        if sum_prop.loc[celltype, state] > max_prop:
            max_prop = sum_prop.loc[celltype, state]
            max_celltype = celltype
    cell_type_labels[state] = max_celltype, round(max_prop, 3)
print(cell_type_labels)

In [ ]:
cell_type_labels

In [ ]:
# Let's find the junctions that are the most differentially spliced between microglia and astrocytes 
scores_all_juncs = []
cellstate1 = 7
cellstate2 = 15

for junc_index in range(juncs_probs.shape[0]):
    a = ALPHA_f[junc_index, [cellstate1,cellstate2]]
    b = PI_f[junc_index, [cellstate1,cellstate2]]
    scores_all_juncs.append(score(a, b).item())

# turn scores_all_juncs into dataframe and add junction_id_index as a column
scores_all_juncs_df = pd.DataFrame(scores_all_juncs, columns = ["score"])
scores_all_juncs_df["junction_id_index"] = junction_ids_conversion.junction_id_index.values
scores_all_juncs_df.sort_values(by="score", ascending=False).head(10)
juncs_test=scores_all_juncs_df.sort_values(by="score", ascending=False).head(2).junction_id_index.values

In [ ]:
juncs_test

In [ ]:
for junc in juncs_test:
    plot_juncObsUsage(junc)
    plot_juncProbs(junc)

In [ ]:
# get one matrix CxC just summing up across all trials and so the higher the number the higher confidence that theya re asigned together 